# Financial Document Analyzer — RAG Pipeline
**Stack:** PyMuPDF · HuggingFace Embeddings · FAISS · Groq LLM

In [ ]:
!pip install pymupdf sentence-transformers faiss-cpu groq -q

In [ ]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {PDF_PATH}")

In [ ]:
import fitz

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for i, page in enumerate(doc):
        text += f"\n--- Page {i+1} ---\n{page.get_text()}"
    doc.close()
    return text

raw_text = extract_text(PDF_PATH)
print(f"Extracted {len(raw_text)} characters")

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(raw_text)
print(f"Total chunks: {len(chunks)}")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
import faiss

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))
print(f"Vectors stored: {index.ntotal}")

In [ ]:
def retrieve(query, k=6):
    vec = embedder.encode([query])
    _, indices = index.search(np.array(vec), k)
    return [chunks[i] for i in indices[0] if i < len(chunks)]

In [ ]:
import groq

GROQ_API_KEY = "gsk_your_key_here"
client = groq.Groq(api_key=GROQ_API_KEY)

def ask_llm(question, context_chunks):
    context = "\n\n".join(context_chunks)
    prompt = f"""You are a financial document assistant.
Answer using ONLY the document excerpt below.
If the answer is not in the excerpt, say "I could not find that in the document."
Be concise and use numbers where available.

--- DOCUMENT EXCERPT ---
{context}
--- END ---

Question: {question}
Answer:"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()

print("Groq client ready.")

In [ ]:
def ask(question):
    print(f"Q: {question}")
    print("-" * 60)
    print(f"A: {ask_llm(question, retrieve(question))}\n")

ask("What is the average monthly surplus?")
ask("How much does Arjun pay for rent each month?")
ask("What are the recurring investment expenses?")
ask("What is the closing balance?")
ask("What is Arjun's total monthly income including freelance?")

In [ ]:
import json

faiss.write_index(index, "bank_statement.index")
with open("chunks.json", "w") as f:
    json.dump(chunks, f)

from google.colab import files
files.download("bank_statement.index")
files.download("chunks.json")
print("Saved and downloaded.")